# EXTRACCIÓN WAVELET

## Extracción Wavelet Discreta (Haar y Daubechies 2)

In [ ]:
import pandas as pd
import numpy as np
import pywt
import os
from scipy.stats import entropy
from tqdm import tqdm

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define las rutas generadas en el paso anterior
BASE_DIR = "ruta/a/tu/espacio/de/trabajo"
INPUT_DIR = os.path.join(BASE_DIR, "normalizacion")
OUTPUT_DIR = os.path.join(BASE_DIR, "caracteristicas")

# Wavelets Discretas a procesar
WAVELETS_DWT = ['haar', 'db2']

# ==========================================
# 2. FUNCIONES MATEMÁTICAS
# ==========================================

def calcular_energia(coeffs):
    """
    Calcula la energía (suma de cuadrados) para cada nivel de descomposición.
    Retorna una lista de valores [Energia_Nivel_1, Energia_Nivel_2, ...].
    """
    energias = []
    for c in coeffs:
        e = np.sum(np.square(c))
        energias.append(e)
    return energias

def calcular_entropia(coeffs):
    """
    Calcula la Entropía de Shannon de la distribución de energía en cada nivel.
    """
    entropias = []
    for c in coeffs:
        E = np.square(c)
        E_sum = np.sum(E)
        if E_sum > 0:
            p = E / E_sum
            ent = entropy(p)
        else:
            ent = 0.0
        entropias.append(ent)
    return entropias

def extraer_caracteristicas_dwt(serie, wavelet_name):
    """Aplica DWT y extrae energía y entropía."""
    # Descomposición al máximo nivel matemático posible
    coeffs = pywt.wavedec(serie, wavelet_name, mode='per')
    
    vec_energia = calcular_energia(coeffs)
    vec_entropia = calcular_entropia(coeffs)
    
    return vec_energia, vec_entropia

# ==========================================
# 3. PROCESAMIENTO PRINCIPAL
# ==========================================

def procesar_extraccion_discreta():
    print("=== INICIANDO EXTRACCIÓN DE CARACTERÍSTICAS (HAAR / DB2) ===")
    
    if not os.path.exists(INPUT_DIR):
        print(f"[ERROR] No existe el directorio de entrada: {INPUT_DIR}")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    archivos = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]
    
    for wavelet in WAVELETS_DWT:
        print(f"\n[WAVELET] Procesando familia: {wavelet.upper()} ...")
        
        datos_energia = []
        datos_entropia = []

        for archivo in tqdm(archivos, desc=f"  Archivos ({wavelet})"):
            ruta_completa = os.path.join(INPUT_DIR, archivo)
            etiqueta_clase = archivo.replace(".csv", "")
            
            try:
                # Cargar serie temporal (Índice=Tiempo, Columnas=IPs)
                df = pd.read_csv(ruta_completa, index_col=0)
                
                for ip in df.columns:
                    serie = df[ip].values
                    
                    # Extracción matemática
                    v_ener, v_entr = extraer_caracteristicas_dwt(serie, wavelet)
                    
                    # Preparar fila Dataset ENERGÍA
                    row_ener = {"ip": ip, "etiqueta": etiqueta_clase, "wavelet": wavelet}
                    for i, val in enumerate(v_ener):
                        row_ener[f"nivel_{i}"] = val
                    datos_energia.append(row_ener)

                    # Preparar fila Dataset ENTROPÍA
                    row_entr = {"ip": ip, "etiqueta": etiqueta_clase, "wavelet": wavelet}
                    for i, val in enumerate(v_entr):
                        row_entr[f"nivel_{i}"] = val
                    datos_entropia.append(row_entr)
                    
            except Exception as e:
                print(f"[ERROR] Fallo en archivo {archivo}: {e}")

        # Guardar Datasets
        print(f"  Guardando resultados para {wavelet}...")
        pd.DataFrame(datos_energia).to_csv(os.path.join(OUTPUT_DIR, f"dataset_energia_{wavelet}.csv"), index=False)
        pd.DataFrame(datos_entropia).to_csv(os.path.join(OUTPUT_DIR, f"dataset_entropia_{wavelet}.csv"), index=False)

    print("\n" + "="*60)
    print("PROCESO DWT COMPLETADO")
    print(f"Datasets generados en: {OUTPUT_DIR}")
    print("="*60)

if __name__ == "__main__":
    procesar_extraccion_discreta()

## Extracción Wavelet Continua (Mexican Hat)

In [ ]:
import pandas as pd
import numpy as np
import pywt
import os
from scipy.stats import entropy
from tqdm import tqdm

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define las rutas generadas en el primer paso
BASE_DIR = "ruta/a/tu/espacio/de/trabajo"
INPUT_DIR = os.path.join(BASE_DIR, "normalizacion")
OUTPUT_DIR = os.path.join(BASE_DIR, "caracteristicas")

# Configuración específica para CWT (Mexican Hat)
WAVELET_NAME = 'mexh'
# Definimos 14 escalas para igualar la dimensión de los vectores Haar/Db2
NUM_ESCALAS = 14
ESCALAS = np.arange(1, NUM_ESCALAS + 1)

# ==========================================
# 2. FUNCIONES MATEMÁTICAS
# ==========================================

def calcular_energia_cwt(coeffs_matrix):
    """
    Calcula la energía por escala sumando la energía a lo largo del eje del tiempo.
    coeffs_matrix shape: (n_escalas, longitud_serie).
    """
    energias = np.sum(np.square(coeffs_matrix), axis=1)
    return energias

def calcular_entropia_cwt(coeffs_matrix):
    """Calcula la Entropía de Shannon de la energía por cada escala."""
    entropias = []
    for fila in coeffs_matrix:
        E = np.square(fila)
        E_sum = np.sum(E)
        if E_sum > 0:
            p = E / E_sum
            ent = entropy(p)
        else:
            ent = 0.0
        entropias.append(ent)
    return entropias

def extraer_caracteristicas_cwt(serie, wavelet_name, escalas):
    """Aplica CWT (Mexican Hat) y extrae métricas de energía y entropía."""
    coeffs, _ = pywt.cwt(serie, escalas, wavelet_name)
    
    vec_energia = calcular_energia_cwt(coeffs)
    vec_entropia = calcular_entropia_cwt(coeffs)
    
    return vec_energia, vec_entropia

# ==========================================
# 3. PROCESAMIENTO PRINCIPAL
# ==========================================

def procesar_mexican_hat():
    print("=== INICIANDO EXTRACCIÓN DE CARACTERÍSTICAS (MEXICAN HAT) ===")
    
    if not os.path.exists(INPUT_DIR):
        print(f"[ERROR] No existe el directorio de entrada: {INPUT_DIR}")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    archivos = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]
    
    datos_energia = []
    datos_entropia = []

    print(f"\n[WAVELET] Procesando familia: {WAVELET_NAME.upper()} (CWT)...")

    for archivo in tqdm(archivos, desc="  Archivos"):
        ruta_completa = os.path.join(INPUT_DIR, archivo)
        etiqueta_clase = archivo.replace(".csv", "")
        
        try:
            df = pd.read_csv(ruta_completa, index_col=0)
            
            for ip in df.columns:
                serie = df[ip].values
                
                # Extracción matemática CWT
                v_ener, v_entr = extraer_caracteristicas_cwt(serie, WAVELET_NAME, ESCALAS)
                
                # Fila Dataset Energía
                row_ener = {"ip": ip, "etiqueta": etiqueta_clase, "wavelet": "mexican_hat"}
                for i, val in enumerate(v_ener):
                    row_ener[f"escala_{i+1}"] = val
                datos_energia.append(row_ener)

                # Fila Dataset Entropía
                row_entr = {"ip": ip, "etiqueta": etiqueta_clase, "wavelet": "mexican_hat"}
                for i, val in enumerate(v_entr):
                    row_entr[f"escala_{i+1}"] = val
                datos_entropia.append(row_entr)
                
        except Exception as e:
            print(f"[ERROR] Fallo en archivo {archivo}: {e}")

    # Guardar Datasets
    print(f"  Guardando resultados para {WAVELET_NAME}...")
    pd.DataFrame(datos_energia).to_csv(os.path.join(OUTPUT_DIR, "dataset_energia_mexicanhat.csv"), index=False)
    pd.DataFrame(datos_entropia).to_csv(os.path.join(OUTPUT_DIR, "dataset_entropia_mexicanhat.csv"), index=False)

    print("\n" + "="*60)
    print("PROCESO CWT COMPLETADO")
    print(f"Archivos generados en: {OUTPUT_DIR}")
    print("="*60)

if __name__ == "__main__":
    procesar_mexican_hat()